In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd

import rasterio

import os
import glob

isd_station_history = 'isd-history.txt'
calfire_admin_shapefile = 'CAL_FIRE_Administrative_Units/CAL_FIRE_Administrative_Units.shp'

calfire_zones_analysis = [
    'Lassen-Modoc Unit',
    'Humbolt-Del Norte Unit',
    'Siskiyou Unit',
    'Shasta-Trinity Unit',
    'Mendocino Unit',
    'Tehama-Glenn Unit',
    'Butte Unit',
    'Nevada-Yuba-Placer Unit'
]

weather_variables = [
    'WND',
    'TMP'
]

weather_file = './2022/'

In [2]:
calfire_admin_zones = gpd.read_file(calfire_admin_shapefile)
# calfire_admin_zones['geometry']

In [3]:
isd_stations = pd.read_fwf(isd_station_history)
isd_stations_ca = isd_stations[isd_stations['ST'] == 'CA']
isd_stations_ca['WBAN'] = isd_stations_ca['WBAN'].astype(str).str.zfill(5)
isd_stations_ca

isd_gdf = gpd.GeoDataFrame(isd_stations_ca,geometry=gpd.points_from_xy(isd_stations_ca.LON,isd_stations_ca.LAT),crs='EPSG:4326')

/tmp/ipykernel_7297/3316361950.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  isd_stations_ca['WBAN'] = isd_stations_ca['WBAN'].astype(str).str.zfill(5)


In [4]:
isd_gdf

,USAF,WBAN,STATION NAME,CTRY,ST,CALL,LAT,LON,ELEV(M),BEGIN,END,geometry
14530,690020,93218,JOLON HUNTER LIGGETT MI,US,CA,KHGT,36.000,-121.233,317.0,19640715,19970401,POINT (-121.233 36)
14531,690020,99999,JOLON HUNTER LIGGETT MI,US,CA,KHGT,36.000,-121.233,317.0,20030702,20030801,POINT (-121.233 36)
14532,690070,93217,FRITZSCHE AAF,US,CA,KOAR,36.683,-121.767,43.0,19600404,19930831,POINT (-121.767 36.683)
14533,690080,99999,TARIN KOWT,AF,CA,KQA7,32.600,65.870,1380.0,20030616,20030624,POINT (65.87 32.6)
14539,690140,93101,EL TORO MCAS,US,CA,KNZJ,33.667,-117.733,116.7,19890101,19971231,POINT (-117.733 33.667)
...,...,...,...,...,...,...,...,...,...,...,...,...
29458,999999,93243,MERCED 23 WSW,US,CA,NaN,37.238,-120.883,23.8,20040325,20250713,POINT (-120.883 37.238)
29459,999999,93245,BODEGA 6 WSW,US,CA,NaN,38.321,-123.075,19.2,20080614,20250713,POINT (-123.075 38.321)
29648,A06854,00115,BIG BEAR CITY AIRPORT,US,CA,KL35,34.264,-116.854,2057.1,20140731,20250713,POINT (-116.854 34.264)
29650,A07049,00320,PETALUMA MUNICIPAL AIRP,US,CA,KO69,38.250,-122.600,27.1,20140731,20250713,POINT (-122.6 38.25)


In [5]:
norcal_stations = gpd.clip(isd_gdf,calfire_admin_zones.to_crs('EPSG:4326'))
norcal_stations['id'] = norcal_stations['USAF'] + norcal_stations['WBAN']
norcal_stations

/home/rpdemilt/miniconda3/envs/rapids-24.10/lib/python3.10/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,USAF,WBAN,STATION NAME,CTRY,ST,CALL,LAT,LON,ELEV(M),BEGIN,END,geometry,id
29436,999999,93115,IMPERIAL BEACH REAM FIE,US,CA,KNRS,32.567,-117.117,7.0,19450501,19721231,POINT (-117.117 32.567),99999993115
18787,722904,23196,BROWN FLD MUNI,US,CA,KSDM,32.567,-116.967,160.0,19780429,19971231,POINT (-116.967 32.567),72290423196
18788,722904,99999,BROWN FLD MUNI,US,CA,KSDM,32.567,-116.967,160.0,20000101,20051231,POINT (-116.967 32.567),72290499999
18794,722909,93115,NAVAL AUXILIARY LANDING,US,CA,KNRS,32.568,-117.117,7.2,19730101,20250713,POINT (-117.117 32.568),72290993115
18786,722904,03178,BROWN FIELD MUNICIPAL A,US,CA,KSDM,32.576,-116.994,158.9,20060101,20250713,POINT (-116.994 32.576),72290403178
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20428,725955,24259,SISKIYOU COUNTY AIRPORT,US,CA,KSIY,41.774,-122.468,806.1,19500101,20250713,POINT (-122.468 41.774),72595524259
29091,999999,24286,CRESCENT CITY FAA AI,US,CA,KCEC,41.780,-124.237,16.8,19491015,19550101,POINT (-124.237 41.78),99999924286
29085,999999,24259,MONTAGUE SISKIYOU COUNT,US,CA,KSIY,41.781,-122.472,805.0,19560101,19660101,POINT (-122.472 41.781),99999924259
20429,725955,99999,SISKIYOU CO,US,CA,KSIY,41.783,-122.467,807.0,20000101,20031231,POINT (-122.467 41.783),72595599999


In [6]:
station_file_list = glob.glob(os.path.join(weather_file,'*.csv'))

norcal_file_list = [file for file in station_file_list if file.split('/')[-1][:-4] in norcal_stations['id'].unique()]
norcal_file_list

dfs = []
for file in norcal_file_list:
    df = pd.read_csv(file,low_memory=False)
    dfs.append(df)

weather_df = pd.concat(dfs)

In [ ]:
weather_df

Index(['STATION', 'DATE', 'SOURCE', 'LATITUDE', 'LONGITUDE', 'ELEVATION',
       'NAME', 'REPORT_TYPE', 'CALL_SIGN', 'QUALITY_CONTROL',
       ...
       'GH1', 'IB2', 'KF1', 'OB1', 'AT6', 'AX4', 'MW4', 'AT7', 'KA3', 'KA4'],
      dtype='object', length=136)

In [21]:
attributes = ['STATION','DATE','LATITUDE','LONGITUDE','NAME','TMP','WND']

weather_df_subset = weather_df[attributes].copy()

wind_atr = weather_df_subset['WND'].str.split(',',expand=True)

weather_df_subset['wnd_spd'] = wind_atr[3].astype(np.float64) / 10
weather_df_subset['dir'] = wind_atr[0].astype(np.float64)
weather_df_subset['UWND'] = weather_df_subset['wnd_spd'] * np.cos(np.deg2rad(weather_df_subset['dir']))
weather_df_subset['VWND'] = weather_df_subset['wnd_spd'] * np.sin(np.deg2rad(weather_df_subset['dir']))
weather_df_subset['dir_qa'] = wind_atr[1]
weather_df_subset['wind_qa'] = wind_atr[4]

wind_atr_pass_qc = wind_atr[wind_atr[1] == '5']
num_samples_pass_qc = len(wind_atr_pass_qc)

print(f'Number of Samples with QC 5 {num_samples_pass_qc}')

wind_atr_pass_qc_non_ncei = wind_atr[wind_atr[1] == '1']
num_samples_pass_qc_non_ncei = len(wind_atr_pass_qc_non_ncei)

print(f'Number of Samples with QC 1 {num_samples_pass_qc_non_ncei}')

Number of Samples with QC 5 1042476
Number of Samples with QC 1 284548


In [22]:
wind_atr

,0,1,2,3,4
0,140,5,N,0036,5
1,250,5,N,0036,5
2,310,5,N,0021,5
3,040,5,N,0015,5
4,350,5,N,0051,5
...,...,...,...,...,...
9384,190,5,N,0051,5
9385,210,5,N,0067,5
9386,210,5,N,0093,5
9387,200,5,N,0098,5


In [23]:
weather_df_subset['UWND']

0      -2.757760
1      -1.231273
2       1.349854
3       1.149067
4       5.022520
          ...   
9384   -5.022520
9385   -5.802370
9386   -8.054036
9387   -9.208988
9388   -8.487049
Name: UWND, Length: 2734267, dtype: float64

In [24]:
weather_df_subset = weather_df_subset[(weather_df_subset['dir_qa'] == '5') & (weather_df_subset['wind_qa'] == '5')]

In [25]:
weather_df_subset

,STATION,DATE,LATITUDE,LONGITUDE,NAME,TMP,WND,wnd_spd,dir,UWND,VWND,dir_qa,wind_qa
0,74718703104,2022-01-01T00:52:00,33.63166,-116.16412,"DESERT RESORTS REGIONAL AIRPORT, CA US","+0161,5","140,5,N,0036,5",3.6,140.0,-2.757760,2.314035,5,5
1,74718703104,2022-01-01T01:52:00,33.63166,-116.16412,"DESERT RESORTS REGIONAL AIRPORT, CA US","+0139,5","250,5,N,0036,5",3.6,250.0,-1.231273,-3.382893,5,5
2,74718703104,2022-01-01T02:52:00,33.63166,-116.16412,"DESERT RESORTS REGIONAL AIRPORT, CA US","+0122,5","310,5,N,0021,5",2.1,310.0,1.349854,-1.608693,5,5
3,74718703104,2022-01-01T03:52:00,33.63166,-116.16412,"DESERT RESORTS REGIONAL AIRPORT, CA US","+0089,5","040,5,N,0015,5",1.5,40.0,1.149067,0.964181,5,5
4,74718703104,2022-01-01T04:52:00,33.63166,-116.16412,"DESERT RESORTS REGIONAL AIRPORT, CA US","+0139,5","350,5,N,0051,5",5.1,350.0,5.022520,-0.885606,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9384,72382023182,2022-12-31T21:51:00,34.62944,-118.08309,"PALMDALE AIRPORT, CA US","+0150,5","190,5,N,0051,5",5.1,190.0,-5.022520,-0.885606,5,5
9385,72382023182,2022-12-31T21:53:00,34.62944,-118.08309,"PALMDALE AIRPORT, CA US","+0150,5","210,5,N,0067,5",6.7,210.0,-5.802370,-3.350000,5,5
9386,72382023182,2022-12-31T22:12:00,34.62944,-118.08309,"PALMDALE AIRPORT, CA US","+0156,5","210,5,N,0093,5",9.3,210.0,-8.054036,-4.650000,5,5
9387,72382023182,2022-12-31T22:53:00,34.62944,-118.08309,"PALMDALE AIRPORT, CA US","+0150,5","200,5,N,0098,5",9.8,200.0,-9.208988,-3.351797,5,5


In [13]:
sample = weather_df_subset.sample(n=40000,random_state=1917,replace=False)[['STATION','DATE','LATITUDE','LONGITUDE','UWND','VWND']]

sample.to_csv('./sample_2021_40k.csv',index=False)

In [26]:
sampled_stations = weather_df_subset['STATION'].sample(n=10,random_state=1917,replace=False)
sample = weather_df_subset[weather_df_subset['STATION'].isin(sampled_stations)][['STATION','DATE','LATITUDE','LONGITUDE','UWND','VWND']]

sample.to_csv('eval_stations.csv',index=False)